# Разметка отзывов через LLM API: сбалансированный сбор классов

Ноутбук размечает отзывы из parquet-чанков через API и собирает сбалансированный датасет: по `TARGET_PER_CLASS` примеров для каждого класса.

Основная логика:
- parquet-файлы читаются **по одному чанку**, весь датасет в память не загружается;
- порядок чанков случайный, но строки внутри чанка **не перемешиваются**;
- промпт оставлен без изменения;
- модель, количество примеров на класс и лимиты меняются в блоке **«Основные настройки»**;
- для каждого класса показывается отдельный progress bar;
- если класс не пополняется `MAX_CHUNKS_WITHOUT_CLASS_HIT` чанков подряд, он отключается от дальнейшего сбора.


## 1. Подключение Google Drive и установка библиотек

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip -q install groq pandas tqdm openpyxl pyarrow

## 2. Импорты

In [ ]:
import os
import glob
import json
import time
import random
from pathlib import Path
from getpass import getpass

import pandas as pd
from tqdm.auto import tqdm
from groq import Groq

## 3. Основные настройки

Здесь меняются главные параметры:
- `MODEL_NAME` — модель, используемая через API;
- `TARGET_PER_CLASS` — сколько примеров собрать для каждого класса;
- `MAX_CHUNKS_WITHOUT_CLASS_HIT` — через сколько чанков без пополнения отключать класс;
- `RANDOM_SEED` — фиксирует случайный порядок чанков;
- `USER_PROMPT` ниже — сам промпт, оставлен без изменения.


In [ ]:
# =========================
# Пути к данным
# =========================
BASE_DIR = "/content/drive/MyDrive/MLops_project/project"

# Файл с уже полученными 200000 случайными примерами
SAMPLED_DATA_PATH = (
    f"{BASE_DIR}/data/processed/"
    "random_100_chunks_2000_examples_seed_42/"
    "sample_100_chunks_2000_each.parquet"
)

# Куда сохранять разметку
LABELED_DIR = f"{BASE_DIR}/data/labeled/wb_feedbacks_llm_labeled_from_sample"
os.makedirs(LABELED_DIR, exist_ok=True)

# =========================
# Колонка с текстом отзыва
# =========================
TEXT_COL = "text"

# =========================
# Режим сбалансированного сбора
# =========================
# Сколько примеров нужно собрать для каждого класса
TARGET_PER_CLASS = 50

# Если класс не пополнялся столько чанков подряд, перестаем пытаться его добирать
MAX_CHUNKS_WITHOUT_CLASS_HIT = 1000

# Защита от слишком долгого запуска. None = пройти все найденные parquet-чанки
MAX_CHUNKS_TO_SCAN = None

# Защита от слишком большого количества API-запросов. None = без ограничения
MAX_REVIEWS_TO_SEND_TO_MODEL = None

# Случайный порядок чанков будет воспроизводимым при одинаковом seed
RANDOM_SEED = 42

# =========================
# API и модель
# =========================
API_KEY_ENV_NAME = "GROQ_API_KEY"
MODEL_NAME = "qwen/qwen3-32b"

# Параметры генерации
TEMPERATURE = 0
MAX_RETRIES = 3
RETRY_BASE_SLEEP_SECONDS = 2

# Пауза между запросами. Если API начнет ругаться на лимиты, поставь 0.5 / 1 / 2.
REQUEST_SLEEP_SECONDS = 0.0

# =========================
# Файлы результата
# =========================
OUTPUT_BASENAME = f"balanced_{TARGET_PER_CLASS}_per_class_random_chunks"
OUTPUT_CSV_PATH = f"{LABELED_DIR}/{OUTPUT_BASENAME}.csv"
OUTPUT_PARQUET_PATH = f"{LABELED_DIR}/{OUTPUT_BASENAME}.parquet"

FINAL_CSV_PATH = f"{LABELED_DIR}/{OUTPUT_BASENAME}_final.csv"
STATUS_CSV_PATH = f"{LABELED_DIR}/{OUTPUT_BASENAME}_class_status.csv"

SAVE_EACH_EXAMPLE_TO_DISK = True
RESET_OUTPUT_FILES_ON_START = False
RESUME_FROM_EXISTING_CSV = True

## 4. API-ключ и клиент

In [ ]:
if API_KEY_ENV_NAME not in os.environ or not os.environ[API_KEY_ENV_NAME]:
    os.environ[API_KEY_ENV_NAME] = getpass(f"Вставь {API_KEY_ENV_NAME}: ")

client = Groq(api_key=os.environ[API_KEY_ENV_NAME])

## 5. Поиск parquet-файлов и случайный порядок чанков

In [ ]:
# Теперь размечаем не все исходные чанки, а один заранее собранный parquet-файл
if not os.path.exists(SAMPLED_DATA_PATH):
    raise FileNotFoundError(f"Не найден файл с выборкой: {SAMPLED_DATA_PATH}")

parquet_files = [SAMPLED_DATA_PATH]
chunk_indices = [0]

print("Файл для разметки:")
print(SAMPLED_DATA_PATH)
print("Будет просмотрено файлов:", len(parquet_files))

Файл для разметки:
/content/drive/MyDrive/MLops_project/project/data/processed/random_100_chunks_2000_examples_seed_42/sample_100_chunks_2000_each.parquet
Будет просмотрено файлов: 1


In [ ]:
sample_df = pd.read_parquet(parquet_files[chunk_indices[0]])

print("Размер первого случайного файла:", sample_df.shape)
print("Колонки:", list(sample_df.columns))

if TEXT_COL not in sample_df.columns:
    raise ValueError(f"В parquet-файлах нет колонки {TEXT_COL}. Колонки: {list(sample_df.columns)}")

sample_df.head()

Размер первого случайного файла: (200000, 5)
Колонки: ['source_chunk_id', 'source_sample_order', 'source_file', 'source_row_idx', 'text']


,source_chunk_id,source_sample_order,source_file,source_row_idx,text
0,1309,0,clean_part_01309.parquet,75721,"Качество хорошее, но мне влики, отказ"
1,1309,0,clean_part_01309.parquet,80184,"Волшебно! Все супер, берите, не пожалеете)"
2,1309,0,clean_part_01309.parquet,19864,❤️❤️❤️❤️❤️ просто прелесть
3,1309,0,clean_part_01309.parquet,76699,Все пучком
4,1309,0,clean_part_01309.parquet,92991,Очень вкусноооо))) Спасибо большое!


## 6. Промпт

Промпт оставлен в том виде, в котором был в исходном файле. Для изменения логики классификации редактируй только содержимое `USER_PROMPT`.


In [ ]:
USER_PROMPT = """
/no_think

Ты выполняешь разметку отзывов покупателей о товарах на маркетплейсе.

Твоя задача — определить, к каким классам относится отзыв. Это multi-label классификация: один отзыв может относиться к одному или нескольким классам.

Важно:

* Размечай только по тексту отзыва.
* Не додумывай факты, которых нет в отзыве.
* Не используй знания о товаре, бренде или маркетплейсе, если они не указаны в тексте.
* Выбирай класс только если в отзыве есть явный или достаточно понятный признак этого класса.
* Нельзя придумывать новые классы.
* Названия классов должны полностью совпадать со списком разрешенных классов.

Разрешенные классы:

1. "Брак / дефект товара"
   Выбирай, если товар сломан, поврежден, не работает, работает неправильно, быстро сломался, имеет трещины, сколы, заводской дефект или явную неисправность.
   Не выбирай этот класс, если товар просто сделан из плохого материала, но не сломан.

2. "Низкое качество материала"
   Выбирай, если покупатель жалуется на плохой материал, дешевый пластик, плохую ткань, кривые швы, хлипкость, слабую сборку, неприятный материал, быструю изнашиваемость.
   Не выбирай этот класс только из-за высокой цены или поломки.

3. "Проблема с размером / посадкой"
   Выбирай, если товар не подошел по размеру, форме, посадке или совместимости: маломерит, большемерит, неудобно сидит, не подошел к устройству, оказался меньше или больше ожидаемого.
   Если в отзыве явно сказано, что в карточке были неверные размеры или характеристики, можно дополнительно выбрать "Несоответствие описанию".

4. "Несоответствие описанию"
   Выбирай, если товар отличается от карточки товара, фото, описания, характеристик или заявленных свойств: другой цвет, другой материал, другая модель, не те характеристики, описание вводит в заблуждение.
   Не выбирай этот класс, если покупатель просто недоволен качеством, но не сравнивает товар с описанием.

5. "Несоответствие ожиданиям / эффекту"
   Выбирай, если товар формально пришел и не сломан, но не выполняет ожидаемую функцию или не дает результата: средство не помогает, устройство слабое, товар бесполезен, эффект отсутствует, результат хуже ожидаемого.
   Не выбирай этот класс, если товар именно неисправен или сломан.

6. "Проблема с комплектацией"
   Выбирай, если в заказе не хватает деталей, аксессуаров, инструкции, кабеля, насадок, креплений или других элементов комплекта; пришла только часть набора; комплект неполный.

7. "Проблема с упаковкой"
   Выбирай, если жалоба относится к упаковке: коробка мятая, упаковка вскрыта, порвана, плохая защита, товар был плохо упакован.
   Если поврежден только внешний вид упаковки, но товар целый, выбирай только этот класс.
   Если сам товар сломан или поврежден, дополнительно выбирай "Брак / дефект товара", если это явно следует из текста.

8. "Проблема доставки / получения"
   Выбирай, если проблема связана с доставкой или получением: долго ехал, задержали, перенесли доставку, пришел не туда, проблемы с пунктом выдачи, курьером, логистикой, заказ потеряли.
   Не выбирай этот класс только из-за мятой коробки — для этого есть "Проблема с упаковкой".

9. "Цена / ценность"
   Выбирай, если покупатель пишет, что товар не стоит своих денег, цена завышена, качество не соответствует цене, можно найти дешевле, за эти деньги ожидал большего.
   Не выбирай этот класс, если в отзыве просто плохое качество без упоминания цены или ценности.

10. "Подделка / оригинальность"
    Выбирай, если покупатель сомневается в подлинности товара или пишет, что товар похож на подделку, неоригинальный, не фирменный, отличается от оригинала, вызывает сомнения бренд, маркировка или упаковка.

11. "Положительный отзыв"
    Выбирай, если покупатель явно доволен товаром: товар понравился, все хорошо, рекомендую, качество устраивает, покупкой доволен, товар оправдал ожидания.
    Этот класс можно сочетать с проблемным классом, если отзыв смешанный: есть и похвала, и конкретная жалоба.
    Не выбирай этот класс, если в отзыве нет явной положительной оценки.

12. "Нейтральный / информационный отзыв"
    Выбирай, если отзыв не содержит явной положительной или отрицательной оценки, а только сообщает факт: получил, пока не использовал, дополню позже, купил в подарок, не открывал, пока ничего сказать не могу.
    Этот класс нельзя сочетать с другими классами.
    Не выбирай этот класс, если есть явная похвала или жалоба.

13. "Другая проблема"
    Выбирай, если в отзыве есть негативная проблема, но ее нельзя уверенно отнести ни к одному из основных классов.
    Этот класс используй только как запасной вариант для непонятной негативной жалобы.
    Не выбирай "Другая проблема", если можно выбрать более конкретный класс.
    Обычно этот класс не нужно сочетать с другими проблемными классами.

Правила выбора классов:

1. Один отзыв может иметь несколько классов.
2. Если отзыв содержит и похвалу, и жалобу, выбери "Положительный отзыв" и соответствующий проблемный класс.
3. "Нейтральный / информационный отзыв" выбирай только тогда, когда нет ни похвалы, ни жалобы.
4. "Нейтральный / информационный отзыв" не должен идти вместе с другими классами.
5. "Другая проблема" выбирай только если есть негатив, но нет подходящего конкретного класса.
6. Если проблема относится к самому товару, выбирай товарный класс: брак, качество, размер, комплектация, описание, эффект, цена, оригинальность.
7. Если проблема относится к процессу получения заказа, выбирай доставку или упаковку.
8. Не добавляй слишком много классов. Каждый выбранный класс должен подтверждаться фрагментом отзыва.
9. При сомнении между конкретным классом и "Другая проблема" выбирай конкретный класс.
10. При сомнении между "Положительный отзыв" и "Нейтральный / информационный отзыв" выбирай "Положительный отзыв" только при явной похвале.
11. Если отзыв очень короткий, например "норм", "ок", "пришло", "получил", и нет понятной оценки, выбирай "Нейтральный / информационный отзыв".
12. Если отзыв короткий, но содержит явную оценку, например "отлично", "супер", "ужас", классифицируй по этой оценке.

Как выставлять confidence:

* 0.90–1.00: класс очевиден, есть прямые слова из отзыва.
* 0.70–0.89: класс достаточно понятен, но формулировка не полностью прямая.
* 0.50–0.69: отзыв неоднозначный, но выбранный класс наиболее вероятен.
* ниже 0.50: используй только если отзыв очень неясный.

Формат ответа:

Верни только валидный JSON-объект без markdown и без текста вне JSON.

JSON должен иметь ровно такие поля:
{
"labels": ["один или несколько классов из разрешенного списка"],
"confidence": 0.0,
"evidence": "короткое объяснение на русском, какие слова из отзыва подтверждают выбор"
}
"""


## 7. Разрешенные классы

In [ ]:
ALLOWED_LABELS = [
    "Брак / дефект товара",
    "Низкое качество материала",
    "Проблема с размером / посадкой",
    "Несоответствие описанию",
    "Несоответствие ожиданиям / эффекту",
    "Проблема с комплектацией",
    "Проблема с упаковкой",
    "Проблема доставки / получения",
    "Цена / ценность",
    "Подделка / оригинальность",
    "Положительный отзыв",
    "Нейтральный / информационный отзыв",
    "Другая проблема",
]

print("Количество классов:", len(ALLOWED_LABELS))

Количество классов: 13


## 8. Функции для промпта, вызова модели и проверки ответа

In [ ]:
import json

def build_full_prompt(review_text):
    """
    Собирает полный запрос к модели:
    - USER_PROMPT содержит правила классификации;
    - ALLOWED_LABELS подставляется автоматически, чтобы не было расхождения;
    - review_text безопасно сериализуется через json.dumps.
    """
    allowed_labels_text = "\n".join(
        f'- "{label}"' for label in ALLOWED_LABELS
    )

    review_text_json = json.dumps(str(review_text), ensure_ascii=False)

    return f"""
{USER_PROMPT}

Разрешенный список классов, который можно использовать в поле labels:
{allowed_labels_text}

Отзыв для классификации:
{review_text_json}

Верни ответ строго в JSON формате:

{{
  "labels": ["..."],
  "confidence": 0.0,
  "evidence": "короткое объяснение, почему выбраны эти классы"
}}

Жесткие требования к ответу:
- Верни только один JSON-объект.
- Не возвращай markdown.
- Не возвращай текст вне JSON.
- Не добавляй комментарии до или после JSON.
- Поле labels должно быть массивом строк.
- labels может содержать только классы из разрешенного списка.
- Если выбрано "Нейтральный / информационный отзыв", это должен быть единственный класс.
- Если выбрано "Другая проблема", не добавляй ее вместе с конкретным проблемным классом.
- confidence — число от 0 до 1.
- evidence — короткое объяснение на русском.
""".strip()

In [ ]:
def normalize_model_response(content):
    """
    Парсит JSON-ответ модели и приводит его к единому формату.
    Также отбрасывает классы, которых нет в ALLOWED_LABELS.
    """
    data = json.loads(content)

    labels = data.get("labels", [])
    if not isinstance(labels, list):
        labels = []

    # Убираем повторы и запрещенные классы, порядок сохраняем
    normalized_labels = []
    for label in labels:
        if label in ALLOWED_LABELS and label not in normalized_labels:
            normalized_labels.append(label)

    if len(normalized_labels) == 0:
        return {
            "labels": [],
            "confidence": 0.0,
            "evidence": "",
            "raw_response": content,
            "error": "no_valid_labels",
        }

    confidence = data.get("confidence", 0.0)
    try:
        confidence = float(confidence)
    except Exception:
        confidence = 0.0
    confidence = max(0.0, min(1.0, confidence))

    return {
        "labels": normalized_labels,
        "confidence": confidence,
        "evidence": str(data.get("evidence", "")),
        "raw_response": content,
        "error": None,
    }

In [ ]:
def classify_review(
    review_text,
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
    max_retries=MAX_RETRIES,
):
    """
    Отправляет один отзыв в LLM API и возвращает результат разметки.

    Чтобы поменять модель, измени MODEL_NAME в блоке настроек
    или передай model_name явно при вызове функции.
    """
    full_prompt = build_full_prompt(review_text)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "/no_think\n"
                            "Ты ассистент для строгой разметки данных для обучения ML-классификатора. "
                            "Твоя цель — высокая точность разметки, а не красивое объяснение. "
                            "Отвечай только валидным JSON-объектом. "
                            "Не добавляй markdown, рассуждения, комментарии или текст вне JSON."
                        ),
                    },
                    {
                        "role": "user",
                        "content": full_prompt,
                    },
                ],
                temperature=temperature,
                response_format={"type": "json_object"},
            )

            content = response.choices[0].message.content
            return normalize_model_response(content)

        except Exception as e:
            if attempt == max_retries - 1:
                return {
                    "labels": [],
                    "confidence": 0.0,
                    "evidence": "",
                    "raw_response": "",
                    "error": str(e),
                }

            time.sleep(RETRY_BASE_SLEEP_SECONDS ** attempt)




In [ ]:
def append_example_to_disk(example_row):
    """
    Сохраняет один найденный пример сразу на диск:
    1) в полный CSV со всеми техническими колонками;
    2) в финальный CSV вида: отзыв | labels | evidence.
    """

    full_row_df = pd.DataFrame([example_row])

    full_row_df.to_csv(
        OUTPUT_CSV_PATH,
        mode="a",
        header=not os.path.exists(OUTPUT_CSV_PATH),
        index=False,
        encoding="utf-8-sig"
    )

    final_row_df = pd.DataFrame([{
        "отзыв": example_row.get("text", ""),
        "labels": example_row.get("labels_str", ""),
        "evidence": example_row.get("evidence", ""),
    }])

    final_row_df.to_csv(
        FINAL_CSV_PATH,
        mode="a",
        header=not os.path.exists(FINAL_CSV_PATH),
        index=False,
        encoding="utf-8-sig"
    )


def save_status_to_disk(class_counts, class_active):
    """
    Сохраняет текущий статус заполнения классов.
    Перезаписывается после каждого чанка.
    """

    class_status_df = pd.DataFrame([
        {
            "label": label,
            "count": class_counts[label],
            "target": TARGET_PER_CLASS,
            "done": class_counts[label] >= TARGET_PER_CLASS,
            "active": class_active[label],
            "stopped": not class_active[label] and class_counts[label] < TARGET_PER_CLASS,
        }
        for label in ALLOWED_LABELS
    ])

    class_status_df.to_csv(
        STATUS_CSV_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    return class_status_df

def load_existing_markup_for_resume():
    """
    Загружает уже сохраненную разметку, чтобы продолжить работу:
    - восстанавливает selected_examples;
    - восстанавливает class_counts;
    - собирает already_labeled_keys, чтобы не размечать те же строки повторно.
    """

    class_counts = {label: 0 for label in ALLOWED_LABELS}
    selected_examples = []
    already_labeled_keys = set()

    if not RESUME_FROM_EXISTING_CSV or not os.path.exists(OUTPUT_CSV_PATH):
        print("Старый CSV с разметкой не найден. Начинаем с нуля.")
        return selected_examples, class_counts, already_labeled_keys

    existing_df = pd.read_csv(OUTPUT_CSV_PATH)

    print("Найдена старая разметка:", OUTPUT_CSV_PATH)
    print("Уже размечено строк:", len(existing_df))

    selected_examples = existing_df.to_dict("records")

    for _, row in existing_df.iterrows():
        source_file = row.get("source_file")
        source_row_idx = row.get("source_row_idx")

        if pd.notna(source_file) and pd.notna(source_row_idx):
            already_labeled_keys.add((str(source_file), int(source_row_idx)))

        labels_counted_str = row.get("labels_counted_str", "")

        if pd.isna(labels_counted_str):
            continue

        labels_counted = [
            label.strip()
            for label in str(labels_counted_str).split(";")
            if label.strip()
        ]

        for label in labels_counted:
            if label in class_counts and class_counts[label] < TARGET_PER_CLASS:
                class_counts[label] += 1

    print("Восстановленные счетчики классов:")
    for label, count in class_counts.items():
        print(f"{label}: {count}/{TARGET_PER_CLASS}")

    return selected_examples, class_counts, already_labeled_keys

## 9. Проверка на одном отзыве

In [ ]:
example_text = str(sample_df.iloc[0][TEXT_COL]).strip()

print(example_text)
classify_review(example_text)

Качество хорошее, но мне влики, отказ


{'labels': ['Другая проблема'],
 'confidence': 0.65,
 'evidence': "Фраза 'мне влики, отказ' указывает на негативную оценку, но неясно, к какому конкретному классу относится проблема.",
 'raw_response': '{\n  "labels": ["Другая проблема"],\n  "confidence": 0.65,\n  "evidence": "Фраза \'мне влики, отказ\' указывает на негативную оценку, но неясно, к какому конкретному классу относится проблема."\n}',
 'error': None}

## 10. Сбалансированная разметка по случайным чанкам

Функция ниже:
- идет по parquet-чанкам в случайном порядке из `chunk_indices`;
- внутри чанка не перемешивает строки;
- отправляет отзывы в модель по одному;
- один отзыв может засчитаться сразу в несколько классов, если модель вернула несколько labels;
- сохраняет отзыв, только если он пополнил хотя бы один еще не заполненный активный класс;
- отключает класс, если он не пополнялся `MAX_CHUNKS_WITHOUT_CLASS_HIT` чанков подряд;
- останавливается, когда все классы собраны до `TARGET_PER_CLASS`, либо когда активных незаполненных классов не осталось.


In [ ]:
def is_collection_finished(class_counts, class_active, target_per_class):
    """
    Возвращает True, если все классы заполнены или если больше нет активных незаполненных классов.
    """
    all_filled = all(class_counts[label] >= target_per_class for label in ALLOWED_LABELS)
    no_active_unfilled = all(
        class_counts[label] >= target_per_class or not class_active[label]
        for label in ALLOWED_LABELS
    )
    return all_filled or no_active_unfilled


def build_class_status_df(class_counts, class_active, chunks_without_hit, target_per_class):
    status_rows = []
    for label in ALLOWED_LABELS:
        if class_counts[label] >= target_per_class:
            status = "filled"
        elif class_active[label]:
            status = "active_not_filled"
        else:
            status = "stopped_no_examples"

        status_rows.append({
            "label": label,
            "count": class_counts[label],
            "target": target_per_class,
            "active": class_active[label],
            "chunks_without_hit": chunks_without_hit[label],
            "status": status,
        })

    return pd.DataFrame(status_rows)

In [ ]:
def label_balanced_by_random_chunks(
    parquet_files,
    chunk_indices,
    text_col=TEXT_COL,
    target_per_class=TARGET_PER_CLASS,
    max_chunks_without_class_hit=MAX_CHUNKS_WITHOUT_CLASS_HIT,
    max_reviews_to_send_to_model=MAX_REVIEWS_TO_SEND_TO_MODEL,
):
    """
    Собирает сбалансированную разметку по классам из parquet-чанков.

    Важно:
    - порядок чанков задается заранее через chunk_indices;
    - строки внутри чанка не перемешиваются;
    - класс отключается, если не пополнялся max_chunks_without_class_hit чанков подряд;
    - один отзыв может пополнить несколько классов, если модель вернула несколько labels.
    """
    if RESET_OUTPUT_FILES_ON_START:
        for path in [OUTPUT_CSV_PATH, FINAL_CSV_PATH, STATUS_CSV_PATH]:
            if os.path.exists(path):
                os.remove(path)

        print("Старые выходные CSV-файлы удалены.")

    selected_examples, class_counts, already_labeled_keys = load_existing_markup_for_resume()

    class_active = {
        label: class_counts[label] < target_per_class
        for label in ALLOWED_LABELS
    }

    chunks_without_hit = {label: 0 for label in ALLOWED_LABELS}
    scanned_chunks = 0
    scanned_reviews = 0
    sent_to_model = 0

    # Один progress bar на каждый класс
    # Один progress bar на каждый класс
    class_bars = {}
    for position, label in enumerate(ALLOWED_LABELS):
        class_bars[label] = tqdm(
            total=target_per_class,
            initial=class_counts[label],
            desc=label[:35],
            position=position,
            leave=True,
        )

        class_bars[label].set_postfix({
            "count": class_counts[label],
            "active": class_active[label],
        })

    chunk_bar_position = len(ALLOWED_LABELS)
    chunk_bar = tqdm(
        chunk_indices,
        desc="Chunks",
        position=chunk_bar_position,
        leave=True,
    )

    try:
        for chunk_idx in chunk_bar:
            if is_collection_finished(class_counts, class_active, target_per_class):
                break

            file_path = parquet_files[chunk_idx]
            scanned_chunks += 1

            df_chunk = pd.read_parquet(file_path)

            if text_col not in df_chunk.columns:
                raise ValueError(
                    f"В файле {file_path} нет колонки {text_col}. "
                    f"Колонки: {list(df_chunk.columns)}"
                )

            # Запоминаем, какие классы пополнились в этом чанке
            labels_hit_in_chunk = set()

            # Внутри чанка идем в исходном порядке: iterrows ничего не перемешивает
            for row_idx, row in df_chunk.iterrows():
                if is_collection_finished(class_counts, class_active, target_per_class):
                    break

                current_key = (Path(file_path).name, int(row_idx))

                if current_key in already_labeled_keys:
                    continue

                if max_reviews_to_send_to_model is not None and sent_to_model >= max_reviews_to_send_to_model:
                    print("Достигнут MAX_REVIEWS_TO_SEND_TO_MODEL")
                    break

                review_text = str(row[text_col]).strip()
                if not review_text or review_text.lower() == "nan":
                    continue

                scanned_reviews += 1
                sent_to_model += 1

                result = classify_review(review_text)
                labels = result.get("labels", [])

                # Засчитываем только те классы, которые еще активны и не заполнены
                labels_to_add = [
                    label
                    for label in labels
                    if label in ALLOWED_LABELS
                    and class_active[label]
                    and class_counts[label] < target_per_class
                ]

                if labels_to_add:
                    for label in labels_to_add:
                        class_counts[label] += 1
                        labels_hit_in_chunk.add(label)
                        class_bars[label].update(1)

                        class_bars[label].set_postfix({
                            "count": class_counts[label],
                            "no_hit_chunks": chunks_without_hit[label],
                            "active": class_active[label],
                        })

                    example_row = {
                        "text": review_text,
                        "labels": labels,
                        "labels_str": "; ".join(labels),
                        "labels_counted": labels_to_add,
                        "labels_counted_str": "; ".join(labels_to_add),
                        "confidence": result.get("confidence", 0.0),
                        "evidence": result.get("evidence", ""),
                        "error": result.get("error"),
                        "raw_response": result.get("raw_response", ""),
                        "source_file": Path(file_path).name,
                        "source_chunk_idx": chunk_idx,
                        "source_row_idx": row_idx,
                    }

                    selected_examples.append(example_row)

                    if SAVE_EACH_EXAMPLE_TO_DISK:
                        append_example_to_disk(example_row)

                    already_labeled_keys.add(current_key)

                if REQUEST_SLEEP_SECONDS > 0:
                    time.sleep(REQUEST_SLEEP_SECONDS)

            # После чанка обновляем счетчики "чанков без пополнения"
            for label in ALLOWED_LABELS:
                if class_counts[label] >= target_per_class:
                    class_active[label] = False
                    class_bars[label].set_postfix({
                        "count": class_counts[label],
                        "status": "filled",
                    })
                    continue

                if not class_active[label]:
                    continue

                if label in labels_hit_in_chunk:
                    chunks_without_hit[label] = 0
                else:
                    chunks_without_hit[label] += 1

                if chunks_without_hit[label] >= max_chunks_without_class_hit:
                    class_active[label] = False
                    class_bars[label].set_postfix({
                        "count": class_counts[label],
                        "status": "stopped",
                        "no_hit_chunks": chunks_without_hit[label],
                    })
                else:
                    class_bars[label].set_postfix({
                        "count": class_counts[label],
                        "no_hit_chunks": chunks_without_hit[label],
                        "active": class_active[label],
                    })

            chunk_bar.set_postfix({
                "saved": len(selected_examples),
                "sent": sent_to_model,
                "min_count": min(class_counts.values()),
                "active": sum(class_active.values()),
            })

            save_status_to_disk(class_counts, class_active)

            if max_reviews_to_send_to_model is not None and sent_to_model >= max_reviews_to_send_to_model:
                break

    finally:
        for bar in class_bars.values():
            bar.close()
        chunk_bar.close()

    labeled_df = pd.DataFrame(selected_examples)
    status_df = build_class_status_df(
        class_counts=class_counts,
        class_active=class_active,
        chunks_without_hit=chunks_without_hit,
        target_per_class=target_per_class,
    )

    stats = {
        "scanned_chunks": scanned_chunks,
        "scanned_reviews": scanned_reviews,
        "sent_to_model": sent_to_model,
        "saved_examples": len(selected_examples),
        "class_counts": class_counts,
    }

    return labeled_df, status_df, stats

## 11. Запуск разметки

In [ ]:
labeled_df, class_status_df, run_stats = label_balanced_by_random_chunks(
    parquet_files=parquet_files,
    chunk_indices=chunk_indices,
    text_col=TEXT_COL,
    target_per_class=TARGET_PER_CLASS,
    max_chunks_without_class_hit=MAX_CHUNKS_WITHOUT_CLASS_HIT,
    max_reviews_to_send_to_model=MAX_REVIEWS_TO_SEND_TO_MODEL,
)

print("Статистика запуска:")
print(json.dumps(run_stats, ensure_ascii=False, indent=2))

class_status_df

Найдена старая разметка: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_llm_labeled_from_sample/balanced_50_per_class_random_chunks.csv
Уже размечено строк: 118
Восстановленные счетчики классов:
Брак / дефект товара: 8/50
Низкое качество материала: 11/50
Проблема с размером / посадкой: 26/50
Несоответствие описанию: 5/50
Несоответствие ожиданиям / эффекту: 9/50
Проблема с комплектацией: 6/50
Проблема с упаковкой: 4/50
Проблема доставки / получения: 3/50
Цена / ценность: 5/50
Подделка / оригинальность: 1/50
Положительный отзыв: 50/50
Нейтральный / информационный отзыв: 5/50
Другая проблема: 6/50


Брак / дефект товара:  16%|#6        | 8/50 [00:00<?, ?it/s]

Низкое качество материала:  22%|##2       | 11/50 [00:00<?, ?it/s]

Проблема с размером / посадкой:  52%|#####2    | 26/50 [00:00<?, ?it/s]

Несоответствие описанию:  10%|#         | 5/50 [00:00<?, ?it/s]

...

Подделка / оригинальность:   2%|2         | 1/50 [00:00<?, ?it/s]

Положительный отзыв: 100%|##########| 50/50 [00:00<?, ?it/s]

Нейтральный / информационный отзыв:  10%|#         | 5/50 [00:00<?, ?it/s]

Другая проблема:  12%|#2        | 6/50 [00:00<?, ?it/s]

Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
labeled_df.head(20)

## 12. Сохранение результата

In [ ]:
# =========================
# Сохранение результатов
# =========================

# Полный CSV уже сохранялся построчно во время разметки.
# Здесь сохраняем итоговый parquet из labeled_df, который вернула функция.
labeled_df.to_parquet(OUTPUT_PARQUET_PATH, index=False)

# Финальный статус классов уже есть в class_status_df,
# который вернула функция label_balanced_by_random_chunks.
class_status_df.to_csv(
    STATUS_CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Сохранено полное CSV:", OUTPUT_CSV_PATH)
print("Сохранено полное Parquet:", OUTPUT_PARQUET_PATH)
print("Сохранен финальный CSV:", FINAL_CSV_PATH)
print("Сохранен статус классов:", STATUS_CSV_PATH)

display(labeled_df.head(20))
display(class_status_df)

In [ ]:
import re
from difflib import SequenceMatcher

def normalize(text):
    text = str(text).lower()
    text = text.replace("ё", "е")
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

text1 = "Джинсы 🔥 классные."
text2 = "Джинсы с хорошей посадкой, что в наше время редкость, раньше подходили только джинсы Зара, эти очень похожи"

similarity = SequenceMatcher(None, normalize(text1), normalize(text2)).ratio()

print(f"Похожесть: {similarity:.3f}")